# Deep Learning with Keras
**Summer of Science 2026 — CS03: Artificial Intelligence and Machine Learning**  
**Mohit Khyalia | IIT Bombay**

---

## Project Overview

TensorFlow/Keras Sequential API for binary tabular classification — introducing the framework, then systematically comparing training hyperparameters. Connects to `neural_network_foundations.ipynb` (same architecture, now with an autodiff framework) and to `regularization_and_feature_selection.ipynb` (Dropout and L2 as regularization).

### Main goals:

- Build and train a Keras Sequential model on tabular data.
- Apply Dropout and EarlyStopping to control overfitting.
- Compare SGD vs Adam, learning rates, batch sizes, and dropout rates.
- Establish the training configuration reused in the Week 8 Spaceship Titanic model.

---

## Setup and Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

tf.random.set_seed(42)
np.random.seed(42)

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.spines.top': False, 'axes.spines.right': False})

## Dataset and Preprocessing

Loan status dataset — same binary classification task used throughout Weeks 5–6. Preprocessed to NumPy arrays here; Keras models operate on arrays rather than DataFrames.

In [2]:
# Binary tabular dataset — binary classification, mixed features
url = 'https://raw.githubusercontent.com/mohitkhyalia1/sos_2026/refs/heads/main/dataset/loan_status_data.csv'
df = pd.read_csv(url).dropna()
df['Loan_Status'] = (df['Loan_Status'] == 'Y').astype(int)

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
for col in ['Gender','Married','Education','Self_Employed','Property_Area']:
    df[col] = le.fit_transform(df[col].astype(str))

X_raw = df.drop(['Loan_ID','Loan_Status'], axis=1, errors='ignore').values.astype(float)
y_raw = df['Loan_Status'].values.astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw)

imputer = SimpleImputer(strategy='median')
scaler  = StandardScaler()
X_train = scaler.fit_transform(imputer.fit_transform(X_train))
X_test  = scaler.transform(imputer.transform(X_test))

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train)

n_features = X_tr.shape[1]
print(f'Train: {X_tr.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

## Utility Functions

A single `build_model` factory with consistent defaults across all experiments. `train_model` wraps EarlyStopping for reproducibility. `plot_history` is the standard training curve format used throughout Week 8.

In [3]:
def build_model(units=(128, 64), dropout=0.3, l2=0.0,
               optimizer='adam', lr=0.001):
    reg = keras.regularizers.l2(l2) if l2 > 0 else None
    opt = (keras.optimizers.Adam(learning_rate=lr) if optimizer == 'adam'
           else keras.optimizers.SGD(learning_rate=lr, momentum=0.9))
    model = keras.Sequential()
    model.add(layers.Input(shape=(n_features,)))
    for u in units:
        model.add(layers.Dense(u, activation='relu', kernel_regularizer=reg))
        if dropout > 0:
            model.add(layers.Dropout(dropout))
    model.add(layers.Dense(1, activation='sigmoid'))
    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])
    return model

def train_model(model, epochs=300, batch_size=32, patience=20):
    es = EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True)
    return model.fit(X_tr, y_tr, validation_data=(X_val, y_val),
                     epochs=epochs, batch_size=batch_size,
                     callbacks=[es], verbose=0)

def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, metric, ylabel in zip(axes, ['loss','accuracy'], ['Loss','Accuracy']):
        ax.plot(history.history[metric],          color='#1F3864', lw=1.5, label='Train')
        ax.plot(history.history[f'val_{metric}'], color='#C00000', lw=1.5, label='Val')
        ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
        ax.set_title(f'{title} — {ylabel}'); ax.legend()
    plt.tight_layout(); plt.show()

## Baseline — Single Hidden Layer

In [4]:
tf.random.set_seed(42)
model_base = build_model(units=(64,), dropout=0.0)
hist_base = model_base.fit(X_tr, y_tr, validation_data=(X_val, y_val),
                            epochs=150, batch_size=32, verbose=0)
_, test_acc = model_base.evaluate(X_test, y_test, verbose=0)
print(f'Baseline — Test accuracy: {test_acc:.4f}')
plot_history(hist_base, 'Baseline (64, no dropout)')

**Observation:**
Validation loss stops improving well before epoch 150 while training loss continues falling — classic overfitting. The gap between train and val curves mirrors the depth-vs-accuracy divergence from `decision_trees_and_ensembles.ipynb`. Dropout and EarlyStopping will close this gap.

## Dropout Regularization

Dropout randomly zeroes a fraction of neurons during each training step. In effect it is L2 regularization applied to activations rather than weights — connecting to `regularization_and_feature_selection.ipynb`.

In [5]:
tf.random.set_seed(42)
model_drop = build_model(units=(128, 64), dropout=0.3)
hist_drop = model_drop.fit(X_tr, y_tr, validation_data=(X_val, y_val),
                            epochs=150, batch_size=32, verbose=0)
_, test_acc_drop = model_drop.evaluate(X_test, y_test, verbose=0)
print(f'With Dropout=0.3 — Test accuracy: {test_acc_drop:.4f}')
plot_history(hist_drop, 'Dropout=0.3 (128-64)')

**Observation:**
Dropout narrows the gap between training and validation curves — the network generalises better because it cannot rely on any single neuron. Validation accuracy is more stable across epochs compared to the baseline.

## EarlyStopping

In [6]:
tf.random.set_seed(42)
model_es = build_model(units=(128, 64), dropout=0.3)
hist_es = train_model(model_es, epochs=500, patience=15)
stopped = len(hist_es.history['loss'])
_, test_acc_es = model_es.evaluate(X_test, y_test, verbose=0)
print(f'Stopped at epoch: {stopped}')
print(f'EarlyStopping — Test accuracy: {test_acc_es:.4f}')
plot_history(hist_es, 'EarlyStopping + Dropout')

**Observation:**
EarlyStopping halts training when validation loss has not improved for 15 consecutive epochs and restores the best weights — removing the need to manually choose the number of epochs. Combined with Dropout it produces the best generalisation of the three configurations.

## Part B — Training Strategy Comparisons

With the basic Keras workflow established, systematically sweeping the main training hyperparameters. Findings from each comparison directly inform the Week 8 configuration choices.

## Optimizer Comparison — SGD vs Adam

Adam adapts the learning rate per parameter using first and second moment estimates. SGD uses a fixed global rate. Connecting to `gradient_descent_visualization.ipynb` (Week 3): SGD is the standard loop; Adam is an improved variant.

In [7]:
tf.random.set_seed(42)
opt_results = []
for label, opt, lr in [
    ('SGD',             'sgd',  0.01),
    ('SGD + momentum',  'sgd',  0.01),
    ('Adam',            'adam', 0.001),
]:
    m = build_model(optimizer=opt, lr=lr)
    if 'momentum' in label:
        # rebuild with momentum
        m.compile(
            optimizer=keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
            loss='binary_crossentropy', metrics=['accuracy'])
    hist = train_model(m, epochs=300)
    _, acc = m.evaluate(X_test, y_test, verbose=0)
    opt_results.append({'Optimizer': label, 'Test Acc': acc, 'Epochs': len(hist.history['loss'])})
    print(f'{label:20s} | epochs: {len(hist.history["loss"]):4d} | test acc: {acc:.4f}')

print(pd.DataFrame(opt_results).to_string(index=False))

**Observation:**
Adam converges in significantly fewer epochs than plain SGD — the adaptive per-parameter learning rates pay off on tabular data. Adam is the default optimizer for the Week 8 Spaceship Titanic model.

## Learning Rate Effect

In [8]:
tf.random.set_seed(42)
lr_results = []
for lr in [0.1, 0.01, 0.001, 0.0001]:
    m = build_model(optimizer='adam', lr=lr)
    hist = train_model(m, epochs=300)
    _, acc = m.evaluate(X_test, y_test, verbose=0)
    lr_results.append({'LR': lr, 'Test Acc': acc, 'Epochs': len(hist.history['loss'])})
    print(f'lr={lr:.4f} | epochs: {len(hist.history["loss"]):4d} | test acc: {acc:.4f}')

lr_df = pd.DataFrame(lr_results)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(len(lr_df)), lr_df['Test Acc'], 'o-', color='#1F3864', lw=1.5)
ax.set_xticks(range(len(lr_df)))
ax.set_xticklabels([str(r) for r in lr_df['LR']])
ax.set_xlabel('Learning Rate')
ax.set_ylabel('Test Accuracy')
ax.set_title('Adam — Learning Rate vs Test Accuracy')
plt.tight_layout()
plt.show()

**Observation:**
lr=0.1 is unstable; lr=0.0001 is too slow to converge within budget. lr=0.001 is the sweet spot for Adam — mirroring the learning rate trade-off from `gradient_descent_visualization.ipynb` (Week 3).

## Dropout Rate Comparison

In [9]:
tf.random.set_seed(42)
drop_results = []
for dr in [0.0, 0.1, 0.2, 0.3, 0.5]:
    m = build_model(dropout=dr)
    hist = train_model(m)
    train_acc = m.evaluate(X_tr,   y_tr,   verbose=0)[1]
    test_acc  = m.evaluate(X_test, y_test, verbose=0)[1]
    drop_results.append({'Dropout': dr, 'Train Acc': train_acc, 'Test Acc': test_acc})
    print(f'dropout={dr} | train: {train_acc:.4f} | test: {test_acc:.4f}')

drop_df = pd.DataFrame(drop_results)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(drop_df['Dropout'], drop_df['Train Acc'], 'o-', color='#1F3864', lw=1.5, label='Train')
ax.plot(drop_df['Dropout'], drop_df['Test Acc'],  'o-', color='#C00000', lw=1.5, label='Test')
ax.set_xlabel('Dropout Rate')
ax.set_ylabel('Accuracy')
ax.set_title('Dropout Rate — Train vs Test Accuracy')
ax.legend()
plt.tight_layout()
plt.show()

**Observation:**
Dropout=0.3 gives the best test accuracy. At 0.5 the model under-regularises on this dataset size. Dropout=0.3 is the setting used in the Week 8 Spaceship Titanic Keras model.

## Batch Size Comparison

In [10]:
tf.random.set_seed(42)
bs_results = []
for bs in [8, 16, 32, 64, 128]:
    m = build_model()
    hist = train_model(m, epochs=300, batch_size=bs)
    _, acc = m.evaluate(X_test, y_test, verbose=0)
    bs_results.append({'Batch Size': bs, 'Test Accuracy': acc, 'Epochs Run': len(hist.history['loss'])})
    print(f'batch={bs:4d} | epochs: {len(hist.history["loss"]):4d} | test acc: {acc:.4f}')

bs_df = pd.DataFrame(bs_results)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(bs_df['Batch Size'].astype(str), bs_df['Test Accuracy'], color='#1F3864', alpha=0.85)
axes[0].set_xlabel('Batch Size'); axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('Batch Size vs Test Accuracy')
axes[1].bar(bs_df['Batch Size'].astype(str), bs_df['Epochs Run'], color='#C00000', alpha=0.85)
axes[1].set_xlabel('Batch Size'); axes[1].set_ylabel('Epochs Run')
axes[1].set_title('Batch Size vs Training Duration')
plt.tight_layout()
plt.show()

**Observation:**
Batch size 32 balances gradient noise and training stability. Larger batches converge faster per epoch but often generalise slightly worse. Batch size 32 is the default for Week 8.

## Best Configuration Summary

In [11]:
print("Recommended training configuration for Week 8 Spaceship Titanic Keras model:")
print("  Optimizer:     Adam (lr=0.001)")
print("  Batch size:    32")
print("  Dropout:       0.3")
print("  EarlyStopping: patience=20, monitor=val_loss, restore_best_weights=True")
print("  Epochs:        500 (EarlyStopping will terminate early)")
print()
print("These settings are applied without further search in Week 8.")

# ReduceLROnPlateau: an alternative to a fixed learning rate
# Halves lr when val_loss does not improve for 8 epochs
# Used in Week 8 deep_learning_keras.ipynb for Spaceship Titanic training
from tensorflow.keras.callbacks import ReduceLROnPlateau
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=8, min_lr=1e-6, verbose=0)
print("ReduceLROnPlateau defined — used in Week 8 alongside EarlyStopping.")

**Observation:**
All five experiments converge on the same conclusion: Adam lr=0.001, dropout=0.3, batch=32, EarlyStopping patience=20 is the reliable default for binary tabular classification at this data scale. On the larger Spaceship Titanic dataset (~8700 rows vs ~490 here), the same configuration applies — larger data benefits neural networks more than it hurts.